# FoxTrack 05: Test of other settings

The baseline results of notebook 02 use two choices: the benign period is divided 70% / 15% / 15% (train / calibration / test), and the moving mean uses the last 100 packets.

This notebook trains the baseline again with a different division (60% / 20% / 20%), and with moving mean windows of 50, 100, and 200 packets. If the conclusions stay the same, they do not depend on one choice.

**Before you run it:** add the output of notebook 01 in **Input > Add Input > Notebook Output**, and set **Internet** to on.

MLflow records each run in the file `mlflow.db`, in the experiment `baseline_settings_test`.

In [ ]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/settings_test"
K_COMPONENTS = 20
SPLITS = ((0.70, 0.15), (0.60, 0.20))   # (train part, calibration part) of the benign period
SMOOTH_WINDOWS = (1, 50, 100, 200)       # 1 = no moving mean
TARGET_FPRS = (1e-3, 1e-4)
ALERT_RULES = ((1, 1), (3, 5))

In [ ]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import anomaly, etl, split
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

## 1. Train and evaluate each division

In [ ]:
%%time
import os, time
import mlflow
import pandas as pd

spark = etl.build_spark(driver_memory="20g")
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("baseline_settings_test")
os.makedirs(OUT_DIR, exist_ok=True)

all_results = []
for train_frac, calib_frac in SPLITS:
    name = f"{round(train_frac * 100)}/{round(calib_frac * 100)}/{round((1 - train_frac - calib_frac) * 100)}"
    for row in split.load_summary(PARQUET_ROOT):
        start = time.time()
        s = split.onset_split(row, train_frac, calib_frac)
        capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
        results, _, _ = anomaly.run_capture(capture, s, K_COMPONENTS, TARGET_FPRS, ALERT_RULES, SMOOTH_WINDOWS)
        results.insert(1, "split", name)
        all_results.append(results)
        with mlflow.start_run(run_name=f"{name}_{s.attack}"):
            mlflow.log_params({"attack": s.attack, "model": "scaler+pca", "k_components": K_COMPONENTS, "split": name,
                               "train_end": s.train_end, "calib_end": s.calib_end, "onset": s.onset})
            for r in results.itertuples():
                tag = f"fpr{r.target_fpr:g}_{r.alert_rule.replace(' ', '_')}"
                mlflow.log_metric(f"false_alarm_events_{tag}", r.false_alarm_events)
                mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
                mlflow.log_metric(f"delay_credible_{tag}", int(r.delay_credible))
                if r.detected:
                    mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
        print(f"{name} / {s.attack}: done in {time.time() - start:.0f} s")

results = pd.concat(all_results, ignore_index=True)
results.to_csv(f"{OUT_DIR}/results.csv", index=False)

## 2. Summary

For each division and each alert rule, at a target false positive rate of 0.0001:

- `credible`: the number of attacks with a credible detection (of 9).
- `zero_events`: the number of recordings with zero false alarm events (of 9).
- `placebo`: the number of false attack starts that the credibility check accepted.

In [ ]:
at = results[results["target_fpr"] == 1e-4]
summary = at.groupby(["split", "alert_rule"]).agg(
    credible=("delay_credible", "sum"),
    zero_events=("false_alarm_events", lambda x: int((x == 0).sum())),
    placebo=("placebo_credible", "sum"),
).reset_index()
summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
summary

**Credible delay in packets** for each attack. An empty cell shows a detection that is not credible.

In [ ]:
at.assign(credible=at["delay_packets"].where(at["delay_credible"])).pivot(index="attack", columns=["split", "alert_rule"], values="credible")

**False alarm events before the attack start.**

In [ ]:
at.pivot(index="attack", columns=["split", "alert_rule"], values="false_alarm_events")

## 3. Keep the output

In [ ]:
print("### SUMMARY"); print(summary.to_csv(index=False))
print("### RESULTS"); print(results.to_csv(index=False))